# 04 - Advanced: lung nodules

**Do this notebook only after you have finished the project** (notebook 03) and downloaded your lung model.

A lung nodule is a small round lesion, a few millimetres wide. Radiologists look for nodules in CT scans
because some of them are early tumours. Here you train a network that segments them.

The code is the one of the project. What changes is the difficulty: lungs are large and dark, and almost any
model finds them. Nodules are tiny, and the result depends on your choices. This is why this notebook is about
**experiments**: you train a first model, then you try to improve it.

| Part | Steps | What you do |
|---|---|---|
| A | STEP 1 to 4 | prepare the data, as in the project |
| B | STEP 5 to 7 | train a first model (the baseline) and look at its mistakes |
| C | STEP 8 to 10 | experiments, and the final test |

As in the project: replace every `...`, use the variable names of the instructions, and compare your result
with "You should see" before going on. The theory is in section 9 of the handbook.

# STEP 0: Setup

Run the cells. The Accelerator must be **GPU T4 x2**.

In [ ]:
!pip install -q monai

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from monai.data import DataLoader, Dataset
from monai.losses import DiceCELoss, DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    RandFlipd,
    RandRotate90d,
    ScaleIntensityRanged,
)
from monai.utils import set_determinism

In [ ]:
DATA_DIR = Path("/kaggle/input/luna16-lung-slices") / "nodules"

set_determinism(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# STEP 1: The data

The folder `nodules` contains small **patches** of 64 x 64 pixels cut from the CT scans, each with one nodule
inside, and the mask of the nodule (1 = nodule, 0 = background). `metadata.csv` has one row per nodule.

1. Read `metadata.csv` with pandas into `metadata` and print how many nodules there are.
2. Take the first row and load its image and its mask with `nibabel`, in `image` and `mask`.
   The files are `images/<nodule_id>.nii.gz` and `labels/<nodule_id>.nii.gz`.
3. Print the shape of `image` and the fraction of pixels of the mask that are nodule (`mask.mean()`).

In [ ]:
# 1. The table of the nodules (hint: pd.read_csv(DATA_DIR / "metadata.csv"))
metadata = ...
print(...)

# 2. Image and mask of the first nodule
nodule_id = metadata["nodule_id"][0]
image = ...     # hint: nib.load(DATA_DIR / "images" / f"{nodule_id}.nii.gz").get_fdata()
mask = ...

# 3. Shape and fraction of nodule pixels
print(...)
print(...)

**You should see** the number of nodules (1186 with the full dataset), the shape `(64, 64)`, and a fraction
of a few percent: almost all the pixels are background.

The next cell shows eight nodules with the border of their mask. Run it.

In [ ]:
plt.figure(figsize=(16, 4.5))
for i in range(8):
    row = metadata.iloc[i]
    img = nib.load(DATA_DIR / "images" / f"{row.nodule_id}.nii.gz").get_fdata()
    msk = nib.load(DATA_DIR / "labels" / f"{row.nodule_id}.nii.gz").get_fdata()
    plt.subplot(1, 8, i + 1)
    plt.title(f"{row.diameter_mm} mm")
    plt.imshow(img.T, cmap="gray", vmin=-1000, vmax=400)
    plt.contour(msk.T, levels=[0.5], colors="red")
    plt.axis("off")
plt.show()

Two things to notice.

- The nodules have very different sizes, and they can be anywhere: free in the lung, attached to the chest
  wall, next to a vessel.
- **The masks are circles.** The dataset only gives the centre and the diameter of each nodule, measured by
  radiologists, so the mask is a disc of that diameter and not the real outline. A model can never agree
  perfectly with these masks. Keep it in mind when you read the Dice.

# STEP 2: Split by patient

A patient can have several nodules. If one nodule of a patient is in the training set and another one of the
same patient is in the test set, the test is no longer on "new patients": the model has already seen that
scanner, that chest, those tissues. So we divide the **patients**, not the nodules.

The cell is already written. Read it, run it, and check the result.

In [ ]:
# The patients, each one only once
patients = sorted(set(metadata["case_id"]))

# 70% / 15% / 15% of the patients
train_patients, other_patients = train_test_split(patients, test_size=0.3, random_state=42)
val_patients, test_patients = train_test_split(other_patients, test_size=0.5, random_state=42)

def files_of(patient_list):
    """The dictionaries {"image": ..., "label": ...} of all the nodules of these patients."""
    rows = metadata[metadata["case_id"].isin(patient_list)]
    return [
        {"image": DATA_DIR / "images" / f"{nodule_id}.nii.gz", "label": DATA_DIR / "labels" / f"{nodule_id}.nii.gz"}
        for nodule_id in rows["nodule_id"]
    ]

train_files = files_of(train_patients)
val_files = files_of(val_patients)
test_files = files_of(test_patients)

print("patients:", len(train_patients), len(val_patients), len(test_patients))
print("nodules :", len(train_files), len(val_files), len(test_files))

**You should see** two lines of three numbers. The nodules are more than the patients.

# STEP 3: Transforms

Simpler than in the project: the patches already have the same size and the masks already contain 0 and 1.

1. Complete `transforms` with three steps: load image and label, add the channel, rescale the image from
   -1000 / 400 HU to 0 / 1.

For now we do not use augmentation: it will be one of the experiments.

In [ ]:
# 1. The three steps (hint: STEP 4 of the project, without Lambdad and Resized)
transforms = Compose([
    ...
])

check = transforms(train_files[0])
print("image:", check["image"].shape, "min", check["image"].min().item(), "max", check["image"].max().item())
print("label:", check["label"].shape, "values", np.unique(check["label"]))

**You should see**

```
image: torch.Size([1, 64, 64]) min ... max ...     (two numbers between 0 and 1)
label: torch.Size([1, 64, 64]) values [0. 1.]
```

# STEP 4: Datasets and loaders

1. Create `train_ds`, `val_ds` and `test_ds` with `Dataset(data=..., transform=transforms)`.
2. Create `train_loader` (`batch_size=32`, `shuffle=True`, `num_workers=2`) and `val_loader`
   (`batch_size=32`, `shuffle=False`, `num_workers=2`).

In [ ]:
# 1. Datasets
train_ds = ...
val_ds = ...
test_ds = ...

# 2. Loaders
train_loader = ...
val_loader = ...

batch = next(iter(train_loader))
print(batch["image"].shape)

**You should see** `torch.Size([32, 1, 64, 64])`.

# STEP 5: Two functions to train many times

In this notebook you will train many models. To avoid copying the loop every time, it is wrapped in two
functions. They are already written: read them, there is nothing new.

- `new_model()` creates a fresh, untrained U-Net. It is smaller than the one of the project, because the
  images are smaller.
- `train(model, loss_function, train_loader)` is the loop of STEP 9 of the project. It returns the list of the
  validation Dice of each epoch and, at the end, leaves in `model` the weights of its best epoch.

In [ ]:
def new_model(channels=(16, 32, 64, 128), strides=(2, 2, 2)):
    return UNet(
        spatial_dims=2,
        in_channels=1,
        out_channels=2,
        channels=channels,
        strides=strides,
        num_res_units=2,
    ).to(device)


def train(model, loss_function, train_loader, epochs=40, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    dice_metric = DiceMetric()
    val_dices = []
    best_dice = -1

    for epoch in range(epochs):

        # ---------- training ----------
        model.train()
        for batch in train_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

        # ---------- validation ----------
        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)
                preds = model(images).argmax(dim=1, keepdim=True)
                dice_metric(y_pred=preds, y=labels)
        val_dice = dice_metric.aggregate().item()
        dice_metric.reset()
        val_dices.append(val_dice)

        # ---------- remember the best weights ----------
        if val_dice > best_dice:
            best_dice = val_dice
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    print(f"best validation Dice: {best_dice:.3f} at epoch {int(np.argmax(val_dices)) + 1} of {epochs}")
    return val_dices

# STEP 6: The baseline

The **baseline** is the first, simplest model. Every experiment will be compared with it.

1. Create `model` with `new_model()`.
2. Create `loss_function`: the same Dice loss of the project.
3. Train: `val_dices = train(model, loss_function, train_loader)`.
4. Save the best validation Dice in the dictionary `results`, under the name `"baseline"`.
5. Plot `val_dices`.

In [ ]:
results = {}      # name of the experiment -> best validation Dice

set_determinism(seed=42)

# 1. A new model
model = ...

# 2. The loss (hint: DiceLoss(to_onehot_y=True, softmax=True))
loss_function = ...

# 3. Train
val_dices = ...

# 4. Save the result (hint: results["baseline"] = max(val_dices))
...

# 5. Plot val_dices
...

**You should see** the line `best validation Dice: ...` and a curve that goes up, much less smoothly than the
one of the lungs. The best validation Dice is around 0.8 - 0.85: clearly lower than for the lungs, as expected.

Keep the model in a separate variable, so that the experiments do not overwrite it:

In [ ]:
baseline_model = model

# STEP 7: Where does the baseline fail?

Before trying to improve a model, look at its mistakes. The cell computes the Dice of each validation nodule
and plots it against the diameter of the nodule. Then it shows the eight worst cases: true mask in red,
prediction in yellow.

1. Run the cell.
2. Answer the three questions below it.

In [ ]:
def dice_per_nodule(model, dataset):
    """Dice of each sample of a dataset, and the predicted masks."""
    dices, preds = [], []
    model.eval()
    with torch.no_grad():
        for sample in dataset:
            pred = model(sample["image"][None].to(device)).argmax(dim=1)[0].cpu()
            true = sample["label"][0]
            dices.append((2 * (pred * true).sum() / (pred.sum() + true.sum())).item())
            preds.append(pred)
    return np.array(dices), preds


val_nodule_dices, val_preds = dice_per_nodule(baseline_model, val_ds)
val_diameters = metadata[metadata["case_id"].isin(val_patients)]["diameter_mm"].values

plt.figure(figsize=(6, 4))
plt.scatter(val_diameters, val_nodule_dices)
plt.xlabel("diameter of the nodule (mm)")
plt.ylabel("Dice")
plt.show()

worst = np.argsort(val_nodule_dices)[:8]
plt.figure(figsize=(16, 4.5))
for k, i in enumerate(worst):
    sample = val_ds[i]
    plt.subplot(1, 8, k + 1)
    plt.title(f"{val_diameters[i]} mm, Dice {val_nodule_dices[i]:.2f}")
    plt.imshow(sample["image"][0].T, cmap="gray")
    plt.contour(sample["label"][0].T, levels=[0.5], colors="red")
    if val_preds[i].sum() > 0:
        plt.contour(val_preds[i].T, levels=[0.5], colors="yellow")
    plt.axis("off")
plt.show()

Questions:

1. Which nodules have the lowest Dice: the small ones or the large ones? Why would a border one pixel off
   matter more for a small nodule?
2. In the worst cases, what does the model do: it finds nothing, it finds the wrong object (a vessel, the
   chest wall), or it finds the nodule but with a different shape?
3. In how many of the worst cases would you say that the prediction is reasonable and the circle is a poor mask?

# STEP 8: How much does the result change by chance?

The weights of a new model are random numbers, and the batches are shuffled at random. Two trainings of the
same model give two different results. Before any experiment we must know how large this difference is:
an "improvement" smaller than it means nothing.

1. Set the seed to 43 with `set_determinism(seed=43)`.
2. Train a new baseline model exactly as in STEP 6, and save its result in `results["baseline, seed 43"]`.
3. Compute `noise`: the absolute difference between the two baseline results.

In [ ]:
# 1. Another seed
...

# 2. The same model and the same loss, trained again
model = ...
loss_function = ...
val_dices = ...
results["baseline, seed 43"] = ...

# 3. The difference (hint: abs(a - b))
noise = ...
print("noise:", noise)

**You should see** a small number, around 0.01. From now on: an experiment that changes
the Dice by less than `noise` has not shown anything.

# STEP 9: Experiments

Now you try to improve the baseline. The rules (handbook, section 9.2):

- **one change at a time**, everything else as in the baseline;
- **write what you expect before you run**;
- compare the result with the baseline and with `noise`.

### Experiment 1 (guided): another loss

The Dice loss looks only at the overlap. With tiny objects it can be unstable: at the beginning the overlap is
almost zero and the network gets little information. `DiceCELoss` adds the cross-entropy, a loss that judges
every pixel one by one.

1. Write in the comment what you expect.
2. Create a new model and the loss `DiceCELoss(to_onehot_y=True, softmax=True)`.
3. Train and save the result in `results["DiceCE loss"]`.

In [ ]:
# 1. I expect: ...

set_determinism(seed=42)

# 2. New model, new loss
model = ...
loss_function = ...

# 3. Train and save the result
val_dices = ...
results["DiceCE loss"] = ...

The next cell prints all the results so far. Run it after each experiment.

In [ ]:
for name, dice in results.items():
    print(f"{dice:.3f}   {name}")
print(f"\nnoise between two identical runs: {noise:.3f}")

**You should see** three lines. Is the difference between "DiceCE loss" and "baseline" larger than the noise?
In which direction? An experiment that does not improve the result is still a result: write it down.

### Experiment 2 (guided): augmentation

A nodule is round: mirrored or rotated by 90 degrees, a patch is still a realistic patch. For the lungs this
was not true. We build a second training loader that applies random flips and rotations.

1. Write in the comment what you expect.
2. Complete `train_transforms_aug`: the three steps of `transforms`, followed by the two random transforms
   already written.
3. Create `train_loader_aug` from `train_files` and `train_transforms_aug` (same settings as `train_loader`).
4. Train a new model with the baseline loss on `train_loader_aug`, and save the result in
   `results["augmentation"]`.

In [ ]:
# 1. I expect: ...

set_determinism(seed=42)

# 2. Transforms with augmentation
train_transforms_aug = Compose([
    ...,      # the three steps of STEP 3
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image", "label"], prob=0.5),
])

# 3. Loader with augmentation
train_loader_aug = DataLoader(Dataset(data=train_files, transform=...),
                              batch_size=32, shuffle=True, num_workers=2)

# 4. Train and save the result
model = ...
loss_function = ...
val_dices = ...
results["augmentation"] = ...

Run the cell that prints the results again.

### Your experiments

Choose from the table, or invent your own. For each experiment copy the cell below, and change one thing.

| Experiment | What to change |
|---|---|
| Learning rate | `train(..., lr=1e-2)` and `train(..., lr=1e-4)` |
| More epochs | `train(..., epochs=80)`: does the best epoch move? |
| Smaller network | `new_model(channels=(8, 16, 32, 64))` |
| Larger network | `new_model(channels=(32, 64, 128, 256))` |
| Narrower window | in the transforms, `a_min=-1000, a_max=200` (then also for validation: build new datasets) |
| Two changes that worked | combine them: is the effect the sum of the two? |
| Less data | `train_files[:len(train_files) // 4]`: how much does the Dice drop? |

When you find a model better than the baseline, look at its mistakes with the cell of STEP 7 (replace
`baseline_model` with your model): did it improve on the small nodules?

In [ ]:
# What I change: ...
# I expect: ...

set_determinism(seed=42)

model = new_model()
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
val_dices = train(model, loss_function, train_loader)
results["my experiment"] = max(val_dices)        # give it a name that says what you changed

# STEP 10: The final test

Choose your best configuration from `results`. Only now we open the test set, once.

1. Train your best configuration again and keep the model in `final_model`.
   (If the baseline is still your best model, use `baseline_model`.)
2. Compute the Dice of each test nodule with `dice_per_nodule(final_model, test_ds)`.
3. Print the mean, and compare it with your best validation Dice.

In [ ]:
# 1. The final model
final_model = ...

# 2. Dice of each test nodule
test_nodule_dices, test_preds = ...

# 3. Mean
print("mean test Dice:", ...)

**You should see** a mean test Dice close to your validation Dice, usually a little lower.

### Check yourself

1. Why is the test Dice usually a little lower than the best validation Dice?
2. A classmate reports "Dice 0.74 with my new loss, against 0.72 of the baseline: the new loss is better".
   What do you ask them?
3. The masks are discs. How does this limit the Dice that any model can reach?
4. Here every patch contains a nodule. What would be different in a real scan?

# To go further

- LUNA16, the challenge on nodule detection this dataset comes from: https://luna16.grand-challenge.org/
- LIDC-IDRI, the original scans, where four radiologists outlined each nodule (and often disagreed):
  https://www.cancerimagingarchive.net/collection/lidc-idri/
- Losses for segmentation, a comparison: https://arxiv.org/abs/2005.13449
- nnU-Net, a method that makes these choices automatically: https://arxiv.org/abs/1809.10486
- MONAI losses: https://docs.monai.io/en/stable/losses.html